# 第 7 周练习：置信度感知 QLoRA 定价器（Confidence-aware Pricer）

## 练习目标

用 **QLoRA** 微调开源因果语言模型做商品价格预测，再对比两种推理方式：

- **贪婪解码（greedy decoding）**：每次取概率最高的 token，直接得到一个价格字符串
- **置信度感知加权 top-k**：用 beam search 得到多条候选，按序列分数加权平均价格

最后用 **MAE / RMSE** 做整体评估，并按类目（Category）做误差分析。

## 和本课第 7 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| QLoRA / 4-bit 量化 | `BitsAndBytesConfig` + `prepare_model_for_kbit_training` |
| PEFT LoRA 适配器 | `LoraConfig` / `get_peft_model` |
| SFT 监督微调 | `SFTTrainer` + `SFTConfig` |
| 价格抽取与评估 | 正则抽数字 + `mean_absolute_error` |

## 怎么跑

1. 需要 GPU（Colab / 本地 CUDA）时效果最好；无 GPU 也能加载，但训练会很慢或不可行
2. 可选：在环境变量里放 `HF_TOKEN`，便于登录 Hugging Face
3. 从上到下运行；训练格会真正微调，耗时与算力相关


In [2]:
# ========== 依赖安装：QLoRA / SFT 常用栈 ==========
# 在 Colab 或新 venv 里按需安装；-q 表示安静模式，少刷安装日志
# 逻辑未改：仍是同一条 pip 安装命令
!pip -q install datasets transformers peft trl bitsandbytes accelerate sentencepiece huggingface_hub scikit-learn


pyenv: version `3.12.12' is not installed (set by /Users/andela/projects/llm_engineering/.python-version)
pyenv: pip: command not found

The `pip' command exists in these Python versions:
  2.7.18

Note: See 'pyenv help global' for tips on allowing both
      python2 and python3 to be found.


In [12]:
# ========== 导入：训练、推理、评估要用的库 ==========

# 标准库 math：后面算 RMSE 时开方
import math
# 标准库 os：读环境变量（如 HF_TOKEN）、设 TOKENIZERS_PARALLELISM
import os
# 标准库 re：从模型输出 / completion 里用正则抽价格数字
import re
# defaultdict：按类目汇总绝对误差时，缺键自动给空列表
from collections import defaultdict

# numpy：数值计算（本笔记本里主要被其他库间接使用）
import numpy as np
# PyTorch：模型张量、设备、推理模式
import torch
# F.softmax：把 beam 的 sequences_scores 转成概率权重
import torch.nn.functional as F
# Hugging Face datasets：加载 ed-donner/items_prompts_lite
from datasets import load_dataset
# huggingface_hub.login：有 HF_TOKEN 时登录，便于下模型/数据集
from huggingface_hub import login
# PEFT：LoRA 配置、加载适配器、把底座包成可训练 PeftModel
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
# sklearn 指标：MAE（平均绝对误差）、MSE（再开方得 RMSE）
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Transformers：因果 LM、分词器、4-bit BitsAndBytes 配置
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
# TRL：监督微调（SFT）的配置与 Trainer
from trl import SFTConfig, SFTTrainer

# 关掉 tokenizer 并行警告噪音（不改变训练算法本身）
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [13]:
# ========== 超参数与路径：集中配置，后面单元格只引用常量 ==========

# 底座模型 id（必须保持英文原样，才能从 Hub 拉到正确权重）
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# 课程配套的轻量 prompt/completion 数据集名
DATASET_NAME = "ed-donner/items_prompts_lite"
# 微调产出（LoRA 适配器 + tokenizer）保存目录
OUTPUT_DIR = "week7_qwen_confidence_pricer"

# 单条序列最大长度（token 数）；过长会截断或更吃显存
MAX_SEQ_LENGTH = 512
# 训练子集上限：加速试验；设 0/None 则用全量（此处按原逻辑用 5000）
TRAIN_LIMIT = 5000
# 评估时最多看多少条测试样本
EVAL_LIMIT = 200

# LoRA 秩 r：越大表达力越强、可训练参数越多
LORA_R = 16
# LoRA alpha：缩放系数，常与 r 成比例
LORA_ALPHA = 32
# LoRA dropout：适配器侧随机丢弃，减轻过拟合
LORA_DROPOUT = 0.05

# 训练轮数；演示用 1 epoch
NUM_EPOCHS = 1
# 每设备微批次大小（显存紧就保持小）
BATCH_SIZE = 2
# 梯度累积步数：等效更大 batch（2 * 8 = 16）
GRAD_ACCUM = 8
# 学习率
LEARNING_RATE = 2e-4

# 从环境变量读 Hugging Face token（没有就跳过登录）
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    # 有 token 才 login，避免把密钥写进笔记本
    login(token=HF_TOKEN)

# 打印是否有 CUDA，方便确认训练环境
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    # 有 GPU 时再打印具体设备名
    print("CUDA device:", torch.cuda.get_device_name(0))


GPU available: False


In [14]:
# ========== 数据：加载 prompt/completion，拼成 SFT 用的 text 字段 ==========

# 按 DATASET_NAME 从 Hub 拉取数据集字典（含 train / val 或 validation / test）
dataset = load_dataset(DATASET_NAME)
# 训练集 split
train_dataset = dataset["train"]
# 有的版本叫 val，有的叫 validation：兼容两种键名
eval_dataset = dataset["val"] if "val" in dataset else dataset["validation"]
# 测试集：后面做推理对比用
test_dataset = dataset["test"]

# 若设置了 TRAIN_LIMIT，截断训练/验证规模以加快试验
if TRAIN_LIMIT:
    # 训练：最多 TRAIN_LIMIT 条
    train_dataset = train_dataset.select(range(min(TRAIN_LIMIT, len(train_dataset))))
    # 验证：至少 100，目标约 TRAIN_LIMIT/10，且不超过现有长度
    eval_dataset = eval_dataset.select(range(min(max(100, TRAIN_LIMIT // 10), len(eval_dataset))))

def to_sft_text(example):
    # SFT 常见格式：把 prompt 与 completion 拼成一条 text 字段
    return {"text": example["prompt"] + example["completion"]}

# map 应用到整份 train/eval，得到带 text 的新 Dataset
train_sft = train_dataset.map(to_sft_text)
eval_sft = eval_dataset.map(to_sft_text)

# 打印规模，确认截断生效
print("Train rows:", len(train_sft))
print("Eval rows:", len(eval_sft))
print("Test rows:", len(test_dataset))
print()
# 看一条样例 prompt（截前 500 字符），建立对数据长什么样的直觉
print("Sample prompt:")
print(test_dataset[0]["prompt"][:500])
print()
# completion 通常是价格字符串，后面会用正则抽数字
print("Sample completion:", test_dataset[0]["completion"])


Train rows: 5000
Eval rows: 500
Test rows: 1000

Sample prompt:
What does this cost to the nearest dollar?

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.

Pr

Sample completion: 219.0


In [15]:
# ========== 分词器 + 4-bit 底座 + LoRA 适配器 ==========

# 加载与底座匹配的 tokenizer；trust_remote_code 允许自定义代码（Qwen 需要）
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# 若没有 pad_token，用 eos 顶上，否则 batch padding 会报错
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# 因果 LM 微调常见：右侧 padding
tokenizer.padding_side = "right"

# 默认不量化；有 CUDA 时再启用 4-bit
bnb_config = None
model_kwargs = {"trust_remote_code": True}
if torch.cuda.is_available():
    # BitsAndBytes 4-bit（NF4 + double quant），显著省显存
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model_kwargs["quantization_config"] = bnb_config
    # device_map=auto：让 accelerate 自动把层放到可用设备
    model_kwargs["device_map"] = "auto"

# 加载因果语言模型底座
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
# 训练时关掉 KV cache，省显存、避免与 gradient checkpoint 冲突
base_model.config.use_cache = False

# 量化训练前准备：冻结底座、开启输入梯度等（k-bit training 惯例）
if torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(base_model)

# LoRA：只训练低秩适配器，不改全量权重
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    # 目标模块：注意力投影 + MLP 门控/上下投影（Qwen 风格命名）
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 把 LoRA 挂到底座上，得到可训练的 PeftModel
model = get_peft_model(base_model, lora_config)
# 打印可训练参数占比，确认真的在训适配器而非全量
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [16]:
# ========== SFTTrainer：把数据、模型、超参捆成训练器 ==========

# SFTConfig：TRL 的训练参数（输出目录、batch、学习率、评估节奏等）
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    # 按 step 做验证，而不是只在 epoch 末
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    # 只保留最近 2 个 checkpoint，节省磁盘
    save_total_limit=2,
    # 有 CUDA 才开 fp16 混合精度
    fp16=torch.cuda.is_available(),
    # 不上报到 wandb 等外部平台
    report_to="none",
    # 告诉 Trainer：数据集里哪一列是拼接好的文本
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    # False：整段 text 都算 loss（含 prompt）；True 则只对 completion 算
    completion_only_loss=False,
)

# 组装 Trainer：模型 + 参数 + train/eval 数据 + tokenizer（processing_class）
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_sft,
    eval_dataset=eval_sft,
    processing_class=tokenizer,
)

# 笔记本里直接显示 trainer 对象，便于确认配置已就绪
trainer


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# ========== 训练并保存适配器 ==========
# 有 GPU、准备好花时间微调时再跑本格
# train()：按 sft_config 真正更新 LoRA 权重
trainer.train()
# 把 Peft 适配器权重存到 OUTPUT_DIR
trainer.model.save_pretrained(OUTPUT_DIR)
# 同步保存 tokenizer，方便下次会话 reload
tokenizer.save_pretrained(OUTPUT_DIR)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/andela/projects/llm_engineering/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


In [ ]:
# ========== 可选：新会话里从磁盘重新挂载已保存的 LoRA ==========
# 下面整段保持注释；需要「只推理、不重新训练」时再取消注释
# 注意：标识符必须保持英文，取消注释后才能直接运行

# base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
# fine_tuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
# fine_tuned_model.eval()
# model = fine_tuned_model


In [ ]:
# ========== 推理助手：贪婪价格 vs 置信度加权 top-k ==========

def model_device(model_to_use):
    # 取模型第一个参数所在 device（cuda:0 / cpu），便于把输入搬过去
    return next(model_to_use.parameters()).device

def extract_price(text):
    # 去掉千分位逗号后，抓第一个整数或小数；抓不到返回 None
    match = re.search(r"[-+]?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(match.group()) if match else None

def extract_category(prompt):
    # 从 prompt 里找 "Category: ..." 行，供后面按类目汇总误差
    match = re.search(r"Category:\s*(.+)", prompt)
    return match.group(1).strip() if match else "Unknown"

@torch.inference_mode()
def greedy_price_predict(prompt, model_to_use=None):
    # 默认用 trainer.model；也可传入外部已加载的模型
    model_to_use = model_to_use or trainer.model
    # 编码 prompt，并搬到模型所在设备
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device(model_to_use))
    # do_sample=False：贪婪解码，只生成最多 8 个新 token（价格很短）
    output = model_to_use.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    # 只解码「新生成」那段，去掉 prompt 前缀
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    # 抽不到数字就当 0.0；同时返回原文便于排查
    return extract_price(completion) or 0.0, completion

@torch.inference_mode()
def weighted_topk_price_predict(prompt, k=5, model_to_use=None):
    # 置信度感知：beam search 出 k 条候选，用序列分数加权平均价格
    model_to_use = model_to_use or trainer.model
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device(model_to_use))
    output = model_to_use.generate(
        **inputs,
        max_new_tokens=8,
        num_beams=k,
        num_return_sequences=k,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    # prompt 长度：用来切开每条序列里的 completion
    prompt_len = inputs["input_ids"].shape[1]
    completions = tokenizer.batch_decode(output.sequences[:, prompt_len:], skip_special_tokens=True)
    # sequences_scores → softmax 得到相对概率权重
    probs = F.softmax(output.sequences_scores.float().cpu(), dim=0).tolist()

    candidates = []
    for completion, prob in zip(completions, probs):
        price = extract_price(completion)
        # 只保留抽到正价格的候选
        if price is not None and price > 0:
            candidates.append({"price": price, "prob": prob, "completion": completion.strip()})

    # 全失败：返回 0 与空列表
    if not candidates:
        return 0.0, []

    # 归一化概率后做加权平均价格
    total_prob = sum(c["prob"] for c in candidates)
    weighted_price = sum(c["price"] * (c["prob"] / total_prob) for c in candidates)
    return weighted_price, candidates


In [ ]:
# ========== 评估：MAE/RMSE + 类目误差 + 最差样本 ==========

def evaluate_predictor(name, predictor, dataset_split, size=EVAL_LIMIT):
    # 截取前 size 条，避免全量测试过久
    subset = dataset_split.select(range(min(size, len(dataset_split))))
    actuals = []
    preds = []
    rows = []

    for row in subset:
        # predictor 约定返回 (预测价格, 细节：字符串或候选列表)
        prediction, detail = predictor(row["prompt"])
        # completion 里抽真值价格；抽不到当 0.0
        actual = extract_price(row["completion"]) or 0.0
        actuals.append(actual)
        preds.append(float(prediction))
        # 逐条记录，供类目报告 / 最差案例分析
        rows.append(
            {
                "category": extract_category(row["prompt"]),
                "actual": actual,
                "pred": float(prediction),
                "abs_error": abs(float(prediction) - actual),
                "detail": detail,
                "prompt": row["prompt"],
            }
        )

    # MAE：平均绝对误差；RMSE：均方根误差
    mae = mean_absolute_error(actuals, preds)
    rmse = math.sqrt(mean_squared_error(actuals, preds))
    return {"name": name, "mae": mae, "rmse": rmse, "rows": rows}

def print_category_report(rows, top_n=10):
    # 按 category 收集绝对误差列表
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["category"]].append(row["abs_error"])

    summary = []
    for category, errors in grouped.items():
        # (类目, 平均绝对误差, 样本数)
        summary.append((category, sum(errors) / len(errors), len(errors)))
    # 误差大、样本多的类目排前面，便于优先排查
    summary.sort(key=lambda item: (-item[1], -item[2], item[0]))

    print("Category error report")
    for category, avg_error, count in summary[:top_n]:
        print(f"{category:30} avg_abs_error=${avg_error:8.2f}  n={count}")

def print_worst_rows(rows, top_n=5):
    # 按绝对误差从大到小，看模型最离谱的几条
    rows = sorted(rows, key=lambda row: row["abs_error"], reverse=True)
    for row in rows[:top_n]:
        print("\n---")
        print("Category:", row["category"])
        print("Actual:", round(row["actual"], 2), "Pred:", round(row["pred"], 2), "AbsErr:", round(row["abs_error"], 2))
        print("Prompt snippet:")
        print(row["prompt"][:260] + ("..." if len(row["prompt"]) > 260 else ""))
        print("Detail:", row["detail"])


In [ ]:
# ========== 训练后跑评估：对比 greedy vs weighted top-k ==========
# 下面调用保持英文标识符（取消注释即可运行）
# 理念：同一测试集、两种解码，看 MAE/RMSE 谁更稳

# greedy_result = evaluate_predictor("greedy", greedy_price_predict, test_dataset)
# weighted_result = evaluate_predictor("weighted_topk", weighted_topk_price_predict, test_dataset)
# print(greedy_result["name"], "MAE=", round(greedy_result["mae"], 2), "RMSE=", round(greedy_result["rmse"], 2))
# print(weighted_result["name"], "MAE=", round(weighted_result["mae"], 2), "RMSE=", round(weighted_result["rmse"], 2))
# print_category_report(weighted_result["rows"])
# print_worst_rows(weighted_result["rows"])
